# Inter-Rater Label Comparison

Compare Anna (AB) labels against all other labelers to assess inter-rater reliability.

This notebook uses the `LabelComparison` class from `src.labels.label_comparison`.

**Goals:**
1. Match AB images with all other labelers (handling ±1 day timezone difference)
2. Visualize polygon overlays for each matched image
3. Compute image-level detection metrics (TP, FP, FN, TN) for each comparison
4. Compute polygon-level area overlap metrics (IoU, precision, recall, F1)
5. Summarize overall agreement across all labelers

In [9]:
# Import the comparison class
import sys
sys.path.insert(0, '..')
from src.labels.label_comparison import LabelComparison

## Run Comparison: AB (Ground Truth) vs PS

In [ ]:
# Define paths
irrigation_table_path = '../data/labels/labeled_surveys/random_sample/latest_irrigation_table.csv'
polygons_path = '../data/labels/labeled_surveys/random_sample/latest_polygons.geojson'
image_boundaries_path = '../data/labels/labeled_surveys/random_sample/latest_irrigation_data.geojson'

# Create comparison object comparing AB (ground truth) to all other labelers
# Figures will be automatically saved to the output directory
comparison = LabelComparison(
    irrigation_table_path=irrigation_table_path,
    polygons_path=polygons_path,
    image_boundaries_path=image_boundaries_path,
    gt_operator='AB',
    comparison_operators=['DSB', 'JL', 'KL', 'MV', 'PS'],  # All other labelers
    min_certainty=3,  # Only include polygons with certainty >= 3
    date_tolerance_days=1,
    iou_threshold=0.1,
    output_dir='../outputs/ab_comparisons'  # Save all figures here
)

## Access and Analyze Results

## Generate Detection Metrics Plots

Generate plots on demand for any comparison operator.

In [ ]:
# Generate confusion matrices and detection metrics for all comparison operators
for op in comparison.comparison_operators:
    print(f"\nGenerating detection plots for {op}...")
    comparison.plot_confusion_matrix(op)
    comparison.plot_detection_metrics_bar(op)

## Generate Area Overlap Plots

View polygon-level area overlap metrics and histograms.

In [ ]:
# Generate area overlap histograms for all comparison operators
for op in comparison.comparison_operators:
    print(f"\nGenerating area overlap plots for {op}...")
    comparison.plot_area_histograms(op)

# Access raw data for a specific operator if needed
print("\nExample: Area metrics for PS")
area_metrics = comparison.compute_area_metrics('PS')
print(area_metrics[['site_id', 'n_gt', 'n_comp', 'n_matched', 'precision', 'recall', 'f1']].head())

## Plot Image Comparisons

Plot individual images or all matched images.

In [ ]:
# Plot a specific image
matches = comparison.get_matches('PS')
if len(matches) > 0:
    first_match = matches.iloc[0]
    comparison.plot_image_comparison(first_match['site_id'], first_match['gt_date'])

In [ ]:
# Plot only images where at least one labeler drew polygons
# Since output_dir is set, figures are automatically saved
print(f"\nPlotting images with polygons...")
print(f"Images will be saved to: {comparison.output_dir}")
comparison.plot_images_with_polygons()

# To plot ALL matched images (including those with no polygons):
# comparison.plot_all_images()

In [ ]:
# Print summary statistics for each comparison operator
for op in comparison.comparison_operators:
    comparison.print_summary(op)

## Example: Compare Against Multiple Labelers

When comparing against multiple labelers, each comparison operator gets:
1. Its own confusion matrix
2. Its own detection metrics bar plot
3. Its own area overlap histograms

The image visualizations will show all comparison labelers in the middle panel with different colors.

In [ ]:
# Example: Create a different comparison with different settings
# For example, comparing against only a subset of labelers

# comparison_subset = LabelComparison(
#     irrigation_table_path=irrigation_table_path,
#     polygons_path=polygons_path,
#     image_boundaries_path=image_boundaries_path,
#     gt_operator='AB',
#     comparison_operators=['PS', 'JL'],  # Only PS and JL
#     min_certainty=4,  # Higher certainty threshold
#     date_tolerance_days=1,
#     output_dir='../outputs/ab_ps_jl_high_certainty'
# )
# 
# # Generate plots for this subset
# for op in comparison_subset.comparison_operators:
#     comparison_subset.print_summary(op)
#     comparison_subset.plot_confusion_matrix(op)
#     comparison_subset.plot_detection_metrics_bar(op)
#     comparison_subset.plot_area_histograms(op)
# 
# comparison_subset.plot_images_with_polygons()

## Key Features of the LabelComparison Class

**Object-Oriented Design**
- Create a `LabelComparison` object once with your data and parameters
- Call methods on demand to generate specific plots and metrics
- Caches computed metrics to avoid recomputation

**Core Methods:**
- `get_matches(comp_op)` - Get matched images DataFrame
- `compute_detection_metrics(comp_op)` - Get image-level detection metrics (TP, FP, FN, TN)
- `compute_area_metrics(comp_op)` - Get polygon-level area metrics DataFrame
- `plot_confusion_matrix(comp_op)` - Plot confusion matrix
- `plot_detection_metrics_bar(comp_op)` - Plot detection metrics bar chart
- `plot_area_histograms(comp_op)` - Plot area overlap histograms
- `plot_image_comparison(site_id, gt_date)` - Plot specific image
- `get_images_with_polygons()` - Get list of images where at least one labeler drew polygons
- `plot_images_with_polygons()` - Plot only images with polygons (excludes "both no irrigation" cases)
- `plot_all_images()` - Plot all matched images
- `print_summary(comp_op)` - Print formatted summary statistics

**1. Automatic Figure Saving**
- Set `output_dir` parameter when creating the comparison object
- ALL figures generated will be automatically saved to that directory
- No need to manually save each plot

**2. Certainty Filtering**
- Set `min_certainty` to filter polygons (default = 3)
- Only polygons meeting the threshold are included in analysis

**3. Smart Image Plotting**
- `plot_images_with_polygons()` - Only plots images where at least one labeler drew polygons
- Automatically skips "both no irrigation" cases
- Saves time and focuses on interesting disagreements

**4. Multi-Labeler Support**
- Compare one ground truth against multiple labelers simultaneously
- **Each comparison is computed independently**: Images matched between GT and labeler A don't need to be labeled by labeler B
- **Image legends show ALL comparison operators** even if they didn't label that specific image
- Operators shown as "N/A" if they didn't label the image, or with "(0)" if they labeled but drew no polygons

**5. Image-Level Detection Metrics**
For each comparison, generates:
- **Confusion Matrix**: TP, FP, FN, TN counts
- **Rate Metrics**: False Positive Rate (FPR), False Negative Rate (FNR)
- **Performance Metrics**: Precision, Recall, F1 Score
- **Visualizations**: Confusion matrix heatmap + bar plot

**6. Area Overlap Metrics (FIXED)**
For each comparison, computes:
- **Polygon-level IoU**: How much overlap between matched polygons
- **Polygon Precision**: Of comparison operator's polygons, what fraction matched GT (FIXED)
- **Polygon Recall**: Of GT's polygons, what fraction were found by comparison operator (FIXED)
- **Histograms**: Distribution of metrics across all matched images